# bartorch Library Internals: tensor bridge, BartLinop, and the hot path

This notebook goes beneath the surface of `bartorch.tools` to explain the
execution model, the zero-copy tensor bridge, `BartContext`, and the
`BartLinop` operator algebra.

**Status: Phase 0 complete.**  The Python-layer infrastructure (context,
dispatch, operator algebra, normalisation) is fully functional.  The C++
extension (Phase 1) is not yet built; ops raise `ImportError` when called.
See `agents.md` for the implementation roadmap.


In [ ]:
import torch
import numpy as np
import bartorch
import bartorch.tools as bt
from bartorch.core.tensor import as_complex64, reverse_dims
from bartorch.core.context import bart_context
from bartorch.utils.cfl import readcfl, writecfl
print(f"bartorch version: {bartorch.__version__}")


## 1. Axis Convention and the Zero-Copy Bridge

bartorch uses **C-order** (last index varies fastest), matching NumPy and PyTorch.
BART internally uses Fortran order (first index varies fastest).

The bridge is zero-copy because:

> A C-order `(coils, ny, nx)` array and a Fortran-order `(nx, ny, coils)` array
> occupy **identical bytes** in memory.

The C++ bridge simply reverses the dim list before calling
`register_mem_cfl_non_managed()` — no data movement.

```
bartorch shape: (coils, phase2, phase1, read)   ← C-order (user sees this)
BART internal:  (read,  phase1, phase2, coils)  ← Fortran-order (BART sees this)
```

Two utilities expose this convention:
- `reverse_dims(dims)` — reverse a shape list
- `as_complex64(t)` — cast to complex64, zero-copy if already correct


In [ ]:
# All ops return plain torch.Tensor (complex64, C-order)
ph = bt.phantom([128, 128])
print(f"Type:    {type(ph)}")
print(f"Shape:   {ph.shape}")   # (1, 128, 128) — coils first (C-order)
print(f"Dtype:   {ph.dtype}")
print(f"Is plain torch.Tensor: {type(ph) is torch.Tensor}")

# The axis utilities:
dims = list(ph.shape)             # [1, 128, 128]  bartorch (C-order)
bart_dims = reverse_dims(dims)    # [128, 128, 1]  BART (Fortran-order)
print(f"\nbartorch dims: {dims}")
print(f"BART dims:     {bart_dims}")

# as_complex64 is zero-copy when already complex64:
ph2 = as_complex64(ph)
print(f"\nSame storage: {ph2.data_ptr() == ph.data_ptr()}")


In [ ]:
# Convert to NumPy
arr_np = ph.squeeze().numpy()   # (128, 128) C-order
print(f"NumPy array shape: {arr_np.shape}, dtype: {arr_np.dtype}")

# Pass a plain tensor or numpy array to any bartorch op — 
# the dispatch layer calls as_complex64() and normalises automatically.
t = torch.ones(128, 128, dtype=torch.float32)
kspace = bt.fft(t, axes=(-1, -2))   # float32 cast to complex64 transparently
print(f"fft input float32 → output dtype: {kspace.dtype}")

if torch.cuda.is_available():
    ph_gpu = ph.cuda()
    print(f"GPU tensor device: {ph_gpu.device}")
else:
    print("CUDA not available — running CPU-only")


## 2. The Dispatch Graph

Every bartorch op routes through `bartorch.core.graph.dispatch()`, which
normalises all inputs to complex64 `torch.Tensor` and selects an execution path:

```
dispatch(op_name, inputs, output_dims, **kwargs)
  │
  │  Normalise: cast all inputs to complex64 torch.Tensor
  │
  ├─ C++ extension available?
  │   → Path A (hot path):
  │       register data_ptr() + reversed dims → bart_command() → plain torch.Tensor
  │
  └─ C++ extension not built?
      → Path B (subprocess):
          write CFL to /dev/shm (reversed dims in header, raw C-order bytes)
          → spawn bart subprocess → read output CFL → reverse dims → torch.Tensor
```

Both paths produce identical numerical results and return plain `torch.Tensor`.


In [ ]:
from bartorch.core.graph import dispatch

# dispatch() selects:
#   - Hot path:  C++ extension available + all inputs are BartTensors
#   - Fallback:  BART subprocess with CFL temp files in /dev/shm

# Check which path will be used:
try:
    import bartorch._bartorch_ext as _ext
    print("C++ extension available — hot path active")
except ImportError:
    print("C++ extension not built — using subprocess fallback")
    print("(Install with:  pip install -e .  or  BARTORCH_SKIP_EXT=1 pip install -e .)")

## 3. BartContext — Chained Operations

Without a `BartContext`, each op independently creates and tears down a CFL
registry mini-session: register inputs → run BART → unlink names.

With `bart_context()`, every op inside the `with` block shares the same
thread-local CFL registry session.  This eliminates redundant re-registration
overhead between consecutive ops and keeps all intermediate results as live
`*.mem` handles — so the next op can consume them without a round-trip through
Python memory.

On the hot path (C++ extension), this means consecutive ops execute entirely
inside C without ever returning to Python between calls.

In [ ]:
# Without a context, each op creates its own CFL registry mini-session.
# With bart_context(), all ops in the block share the same session,
# avoiding re-registration overhead for consecutive calls.

with bart_context() as ctx:
    # All three ops share the same in-memory CFL session
    phantom_img = bt.phantom([128, 128])
    kspace = bt.fft(phantom_img, flags=3)
    # The context cleans up the CFL registry on exit

print(f"kspace shape: {kspace.shape}")
print(f"kspace dtype: {kspace.dtype}")

## 4. Full Pipeline in a Context

The cell below runs a complete MRI acquisition-and-reconstruction pipeline
inside a single `BartContext`.  When the C++ extension is built, all three
BART operations execute without returning to Python between calls — the
intermediate k-space and sensitivity arrays are passed directly by their
`*.mem` handles.

In [ ]:
# Complete MRI acquisition + reconstruction pipeline
# (all ops stay in C when the extension is built)

with bart_context():
    # 1. Generate 8-coil phantom k-space
    kspace_mc = bt.phantom([128, 128], kspace=True, ncoils=8)

    # 2. Coil calibration
    sens = bt.ecalib(kspace_mc, calib_size=16)

    # 3. L1-Wavelet compressed sensing reconstruction
    reco = bt.pics(kspace_mc, sens, R='W:7:0:0.005', i=30)

print(f"Pipeline output shape: {reco.shape}")
print(f"Pipeline output dtype:  {reco.dtype}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(bt.ifft(kspace_mc[..., 0], axes=(-1, -2)).squeeze().abs().numpy(), cmap='gray')
axes[0].set_title('Coil 1 (IFFT)')
axes[0].axis('off')

axes[1].imshow(sens[..., 0, 0].abs().numpy(), cmap='viridis')
axes[1].set_title('Coil 1 sensitivity (ESPIRiT)')
axes[1].axis('off')

axes[2].imshow(reco.squeeze().abs().numpy(), cmap='gray')
axes[2].set_title('PICS reconstruction')
axes[2].axis('off')

plt.suptitle('bartorch pipeline: phantom \u2192 ecalib \u2192 pics', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Working with Real CFL Data

`readcfl` returns a NumPy `complex64` array; `writecfl` accepts any array-like.
To feed real data into bartorch ops, convert to a tensor with `torch.as_tensor()`
(zero-copy when the array is already contiguous `complex64`) or let the bartorch
dispatch layer promote it automatically.

The dispatch layer will copy plain `torch.Tensor` inputs to a `BartTensor` when
needed — the copy is unavoidable if the layout does not match, but it happens at
most once per input per op call.

In [ ]:
# If you have BART's demo data (k-space saved as CFL files), you can load
# them directly and pass to bartorch ops.
#
# Example (replace paths with your actual data):
#
#   kspace = readcfl('/path/to/kspace')          # numpy array, complex64
#   kspace_bt = torch.as_tensor(kspace)          # promote to BartTensor
#   sens = bt.ecalib(kspace_bt, calib_size=24)
#   reco = bt.pics(kspace_bt, sens, lambda_=0.01, wav=True)

# For demonstration, we create synthetic data instead:
synth = np.random.randn(64, 64, 1, 4).astype(np.complex64)
synth += 1j * np.random.randn(64, 64, 1, 4).astype(np.float32)

import tempfile, os
with tempfile.TemporaryDirectory() as tmpdir:
    p = os.path.join(tmpdir, 'synth_kspace')
    writecfl(p, synth)
    loaded = readcfl(p)
    print(f"CFL round-trip: {synth.shape} \u2192 wrote \u2192 loaded {loaded.shape}")
    print(f"Max abs error:  {np.abs(synth - loaded).max():.2e}")

## 6. Linear Operators — Phase 0 Status

BART has a powerful internal linear operator framework (`linop_s*`).
In Phase 0, `BartLinop` provides the full Python-level operator algebra
(composition, adjoint, normal, scalar scaling, sum) without requiring
the C++ extension.  Concrete linop factories (FFT, NUFFT, sensitivity
encoding, wavelets) are planned for Phases 2–3.


In [ ]:
## 6. Linear Operators — BartLinop algebra

`BartLinop` is the public handle type for BART linear operators (`linop_s*`).
It exposes an algebra via Python magic methods — no standalone `chain()` or
`plus()` factory functions in the public API:

```python
from bartorch.ops import BartLinop

# Adjoint (swaps ishape / oshape):
AH = A.H

# Normal operator A^H A (maps domain to itself):
AHA = A.N

# Composition (apply B first, then A):
C = A @ B          # returns BartLinop

# Application:
y = A @ x          # x is torch.Tensor → returns torch.Tensor
y = A(x)           # equivalent

# Sum:
D = A + B          # returns BartLinop

# Scalar multiplication:
E = 2.0 * A        # returns BartLinop

# Shapes are in C-order:
print(A.ishape)    # e.g. (8, 256, 256) — coils, ny, nx
print(A.oshape)    # e.g. (8, 1, 256, 256) — coils, nz, ny, nx
```

The operator algebra works **at Python level** today (no C++ extension needed).
Application (`A(x)`, `A @ x`) requires the C++ extension (Phase 3).


## Summary

### Execution path decision table

| Condition | Path | Notes |
|-----------|------|-------|
| C++ ext built | **Hot path** (Path A) | Zero copies, bart_command() in-process |
| C++ ext not built | **Subprocess fallback** (Path B) | CFL pairs in `/dev/shm` (Linux) or temp dir |

Both paths accept and return plain `torch.Tensor`.

### Key utilities

| Symbol | What it does |
|--------|--------------|
| `as_complex64(t)` | Cast to complex64, zero-copy if already correct |
| `reverse_dims(dims)` | Swap C-order ↔ Fortran-order dim list |
| `BartLinop.H` | Adjoint operator |
| `BartLinop.N` | Normal operator A^H A |
| `A @ B` | Operator composition |
| `A + B` | Operator sum |
| `scalar * A` | Scalar multiplication |

### Roadmap

| Phase | Feature | Status |
|-------|---------|--------|
| 1 | Build system + C++ extension core | planned |
| 2 | FFT, NuFFT, Phantom, ecalib, PICS via C++ | planned |
| 3 | BartLinop + encoding operators | planned |
| 4 | Regularizers + proximal operators | planned |
| 5 | Iterative algorithms | planned |
| 6 | Nonlinear operators | planned |
